<a href="https://colab.research.google.com/github/Mimikaymint/Team_Berlin/blob/main/Flu_Shot/Data/Sprint%201/FLU_SHOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# #            PACKAGES             #
###################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import logging
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from scipy import stats
from sklearn import preprocessing
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import os




In [ ]:
os.listdir()


In [ ]:
##################################################
# T1.1.2. LOADING DATASET & EXAMINIG THE DATASET #
##################################################
from google.colab import files
uploaded = files.upload()

flu_df = pd.read_csv('training_set_features.csv')
flu_df2 = pd.read_csv('training_set_labels.csv')


In [ ]:
import sys
print(sys.executable)


In [ ]:
flu_df = pd.merge(flu_df, flu_df2, on='respondent_id', how='outer')


In [ ]:
###################################
#        DIMENSIONS & INFO        #
###################################
print(flu_df.shape)
flu_df.describe()
print(f"Dataset shape: {flu_df.shape}")
print(flu_df.dtypes)


In [ ]:
###################################
#         MISSING VALUES          #
###################################
print(flu_df.isnull().values.any())
print(flu_df.isnull().sum())

msno.bar(flu_df)
plt.show()


In [ ]:
missing_values = (flu_df.isnull().sum() / len(flu_df)) * 100
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)

plt.figure(figsize=(8,5))
missing_values.plot(kind='bar', color='red')
plt.ylabel("Percentage of Missing Values")
plt.title("Percentage of Missing Values per Column")
plt.show()


In [ ]:
###################################
#       PROFILE SUMMARY           #
###################################
import sweetviz as sv

report = sv.analyze(flu_df)
report.show_html("dataset_profile.html")



In [ ]:
##################################################
# T1.1.3. SUMMARY STATS FOR FEATURES             #
##################################################
cat_cols = flu_df.select_dtypes(include=['object', 'category']).columns
num_cols = flu_df.select_dtypes(include=['int64', 'float64']).columns

numeric_sum = flu_df[num_cols].describe().T
print("Summary statistics for numerical features:")
display(numeric_sum)

In [ ]:
categorical_sum = flu_df[cat_cols].describe().T
print("\nSummary statistics for categorical features:")
display(categorical_sum)

In [ ]:
##################################################
#    VARIABLE DISTRIBUTIONS AND CLASS BALANCE    #
##################################################

print(flu_df['h1n1_vaccine'].value_counts(normalize=True) * 100)
h1n1_dist = flu_df['h1n1_vaccine'].value_counts().reset_index()
h1n1_dist.columns = ['Class', 'Count']
h1n1_dist['Percentage'] = (h1n1_dist['Count'] / h1n1_dist['Count'].sum()) * 100
print(h1n1_dist)


In [ ]:
print(flu_df['seasonal_vaccine'].value_counts(normalize=True) * 100)
seaonal_dist = flu_df['seasonal_vaccine'].value_counts().reset_index()
seaonal_dist.columns = ['Class', 'Count']
seaonal_dist['Percentage'] = (seaonal_dist['Count'] / seaonal_dist['Count'].sum()) * 100
print(seaonal_dist)

In [ ]:
targets = ['h1n1_vaccine', 'seasonal_vaccine']

colors = ['#1f77b4', '#d3d3d3']
for col in targets:
    counts = flu_df[col].value_counts()

    counts = counts.sort_index()

    labels = ['No Flu Shot (0)', ' Flu Shot (1)']
    explode = [0.05] * len(counts)

    plt.figure(figsize=(6,6))
    wedges, texts, autotexts = plt.pie(
        counts,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors[:len(counts)],
        explode=explode,
        shadow=True
    )

    plt.title(f"Distribution of {col}", fontsize=14, fontweight='bold')
    plt.legend(wedges, labels, title="Classes", loc="upper right", fontsize=10)
    plt.show()

In [ ]:
from nbconvert import MarkdownExporter
import nbformat

with open("Flu_Shot.ipynb") as f:
    nb = nbformat.read(f, as_version=4)

exporter = MarkdownExporter()
exporter.exclude_input = True
exporter.exclude_output_prompt = True

body, resources = exporter.from_notebook_node(nb)

with open("Flu_Shot_outputs.md", "w") as f:
    f.write(body)

print("Markdown file created with outputs only!")


In [ ]:
#Task 1.2: Data Cleaning and Preprocessing
#Ticket 1.2.1: Handle missing values
cat_cols = flu_df.select_dtypes(include=['object', 'category']).columns
num_cols = flu_df.select_dtypes(include=['int64', 'float64']).columns.drop('respondent_id', errors='ignore')
imputation_cols = num_cols.tolist() + cat_cols.tolist()
# 1. Implement Imputation Strategies
# Numeric (int/float): Median imputation is applied to ALL numerical/binary columns for consistency
num_imputer = SimpleImputer(strategy="median")
flu_df[num_cols] = num_imputer.fit_transform(flu_df[num_cols])

# Categorical (object/category): Most frequent value (mode) is used
cat_imputer = SimpleImputer(strategy="most_frequent")
flu_df[cat_cols] = cat_imputer.fit_transform(flu_df[cat_cols])

# 2. Verification Code for Imputation
print("\nVerification of Imputation (Checking remaining NaNs):")
total_na_after_imputation = flu_df[imputation_cols].isnull().sum().sum()

if total_na_after_imputation == 0:
    print("SUCCESS: All missing values in numerical and categorical features have been imputed.")
else:
    print(f"ERROR: {total_na_after_imputation} missing values remaining after imputation.")

In [ ]:
#To check for missing values
print(flu_df.isnull().values.any())
print(flu_df.isnull().sum())

In [ ]:
#Ticket 1.2.2: Convert categorical variables to appropriate formats
# 1. Define Ordinal Categories and Features
# Create reusable ordered mappings for OrdinalEncoder
ordinal_features = ['age_group', 'education', 'income_poverty']
age_group_order = ['18 - 34 Years', '35 - 44 Years', '45 - 54 Years', '55 - 64 Years', '65+ Years']
education_order = ['< 12 Years', '12 Years', 'Some College', 'College Graduate']
income_poverty_order = ['Below Poverty', '<= $75,000, Above Poverty', '> $75,000']
ordinal_categories = [age_group_order, education_order, income_poverty_order]

# 2. Define Nominal Features
all_cat_cols_imputed = flu_df.select_dtypes(include=['object', 'category']).columns
nominal_features = [col for col in all_cat_cols_imputed if col not in ordinal_features]

# 3. Create the binary_features
binary_features = [
    'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask',
    'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home',
    'behavioral_touch_face', 'doctor_recc_h1n1', 'doctor_recc_seasonal',
    'chronic_med_condition', 'child_under_6_months', 'health_worker', 'health_insurance']
scaling_features = [col for col in num_cols if col not in binary_features]
# 4. Create the ColumnTransformer
column_transformer = ColumnTransformer(
    transformers=[
        # Ordinal Encoding: For ordered features
        ('ord', OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_features),

        # One-Hot Encoding: For nominal features
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_features),

        # Passthrough: Capped Numerical Features (for subsequent scaling)
        ('num_scale', 'passthrough', scaling_features),

        # Passthrough: Binary Features (includes all 'behavioral_' columns)
        ('binary_passthrough', 'passthrough', binary_features)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
print("ColumnTransformer created for reusable encoding for Ordinal & One-Hot")

In [ ]:
# Check for NaNs in the data
if flu_df.isna().any().any():
    print("\nThere are missing values in the data.")
else:
    print("\nNo missing values was found.")

In [ ]:
#Ticket 1.2.3: Visualize outliers
# Visualization: Use boxplots to visualize outliers in a few key numerical
viz_cols = ['h1n1_concern', 'h1n1_knowledge', 'opinion_h1n1_vacc_effective', 'household_adults', 'household_children']

plt.figure(figsize=(10, 3))
for i, col in enumerate(viz_cols):
    plt.subplot(1, 5, i + 1)
    sns.boxplot(y=flu_df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

# Statistical Method: IQR rule (1.5 * IQR)
# Apply outlier capping ONLY to scaling_features (non-binary columns)
for col in scaling_features:
    Q1 = flu_df[col].quantile(0.25)
    Q3 = flu_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers_count = ((flu_df[col] < lower_bound) | (flu_df[col] > upper_bound)).sum()

    if outliers_count > 0:
        # Treatment Strategy: Capping (Winsorization)
        flu_df[col] = np.where(flu_df[col] < lower_bound, lower_bound, flu_df[col])
        flu_df[col] = np.where(flu_df[col] > upper_bound, upper_bound, flu_df[col])
        # Documentation of Impact:
        print(f"Capped {outliers_count} outliers in column '{col}' using IQR method (bounds: {lower_bound:.2f}, {upper_bound:.2f}).")
    else:
        print(f"No significant outliers found for capping in column '{col}'.")

In [ ]:
#Ticket 1.2.4: Normalize/standardize numerical features if appropriate
scaler = StandardScaler()

In [ ]:
# Ticket 1.2.5: Create a data cleaning pipeline that can be reused for test data

# The Pipeline combines encoding/passthrough (ColumnTransformer) with scaling (StandardScaler)
preprocessing_pipeline = Pipeline(steps=[
    ('encoder', column_transformer),
    ('scaler', scaler)
])
print("preprocessing_pipeline is created.")

In [ ]:
########################################
# Ticket 1.3 Exploratory Visualization #
########################################

# 1.3.1
import matplotlib.pyplot as plt

targets = ['h1n1_vaccine', 'seasonal_vaccine']
colors = ['#1f77b4', '#d3d3d3']

for col in targets:
    counts = flu_df[col].value_counts().sort_index()

    labels = ['No Flu Shot (0)', 'Flu Shot (1)']
    explode = [0.05] * len(counts)

    plt.figure(figsize=(6, 6))
    wedges, texts, autotexts = plt.pie(
        counts,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors[:len(counts)],
        explode=explode,
        shadow=True
    )

    plt.title(f"Distribution of {col}", fontsize=14, fontweight='bold')
    plt.legend(wedges, labels, title="Classes", loc="upper right", fontsize=10)
    plt.show()


In [ ]:
from matplotlib_venn import venn2
import matplotlib.pyplot as plt

v1, v2 = targets
h1n1_only = ((flu_df[v1] == 1) & (flu_df[v2] == 0)).sum()
seasonal_only = ((flu_df[v2] == 1) & (flu_df[v1] == 0)).sum()
both = ((flu_df[v1] == 1) & (flu_df[v2] == 1)).sum()

v = venn2(
    subsets=(h1n1_only, seasonal_only, both),
    set_labels=(v1, v2),
    set_colors=("skyblue", '#1f77b4'),
    alpha=0.9
)

overlap_patch = v.get_patch_by_id('11')
if overlap_patch is not None:
    overlap_patch.set_color('lightgrey')

plt.title(f'Overlap Between {v1} and {v2}')
plt.show()

In [ ]:

flu_df['education'] = flu_df['education'].fillna('Unknown')

demographics = ['age_group', 'sex', 'race', 'education']
for col in demographics:
    flu_df[col] = flu_df[col].astype('category')

demo_df = flu_df.melt(
    id_vars=demographics,
    value_vars=targets,
    var_name='Vaccine',
    value_name='Vaccinated'
)

demo_df_vaccinated = demo_df[demo_df['Vaccinated'] == 1]

for demo in demographics:
    plt.figure(figsize=(8,5))
    ax = sns.countplot(
        data=demo_df_vaccinated,
        x=demo,
        hue='Vaccine',
        palette=['#1f77b4', '#d3d3d3']
    )
    plt.ylabel('Number of Vaccinated Children')
    plt.title(f'Vaccinated Counts by {demo}')
    plt.xticks(rotation=45)

    for p in ax.patches:
        height = p.get_height()
        ax.text(
            x=p.get_x() + p.get_width()/2,
            y=height + 0.5,
            s=int(height),
            ha='center',
            va='bottom',
            fontsize=10
        )

    plt.legend(title='Vaccine')
    plt.tight_layout()
    plt.show()


In [ ]:
demographics = ['age_group', 'sex', 'race', 'education']
for col in demographics:
    flu_df[col] = flu_df[col].astype('category')

demo_pairs = list(itertools.combinations(demographics, 2))

for vaccine in targets:
    n_rows = len(demo_pairs) // 2 + len(demo_pairs) % 2
    n_cols = 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))

    axes = axes.flatten()

    for i, (demo1, demo2) in enumerate(demo_pairs):
        ct = pd.pivot_table(
            flu_df,
            values=vaccine,
            index=demo1,
            columns=demo2,
            aggfunc='mean',
            fill_value=0
        )
        sns.heatmap(ct, annot=True, fmt=".2f", cmap='Blues', ax=axes[i])
        axes[i].set_title(f'{vaccine} Proportion by {demo1} & {demo2}')
        axes[i].set_ylabel(demo1)
        axes[i].set_xlabel(demo2)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

In [ ]:
def classify_urban_rural(x):
    if x == 'Non-MSA':
        return 'Rural'
    else:
        return 'Urban'

flu_df['census_msa'] = flu_df['census_msa'].apply(classify_urban_rural)

flu_df['census_msa'].value_counts()


In [ ]:

flu_df['UrbanRural'] = flu_df['census_msa']

vaccines = ['h1n1_vaccine', 'seasonal_vaccine']
colors = {'Vaccinated':'purple','Not Vaccinated':'blue'}

for vac in vaccines:
    plot_df = flu_df.groupby(['UrbanRural', vac]).size().reset_index(name='count')

    all_combinations = pd.MultiIndex.from_product([['Urban','Rural'], [0,1]], names=['UrbanRural', vac])
    plot_df = plot_df.set_index(['UrbanRural', vac]).reindex(all_combinations, fill_value=0).reset_index()

    plot_df['Vaccinated_Label'] = plot_df[vac].map({0:'Not Vaccinated', 1:'Vaccinated'})

    plot_df['percentage'] = plot_df.groupby('UrbanRural')['count'].apply(lambda x: x / x.sum() * 100)

    plt.figure(figsize=(6,5))
    ax = sns.barplot(
        data=plot_df,
        x='UrbanRural',
        y='percentage',
        hue='Vaccinated_Label',
        palette=colors
    )

    plt.title(f"{vac.replace('_',' ').title()} Vaccination rates")
    plt.ylabel('Percentage (%)')
    plt.ylim(0,100)
    plt.legend(title='Status')

    for p in ax.patches:
        height = p.get_height()
        ax.text(
            x=p.get_x() + p.get_width()/2,
            y=height + 1,
            s=f"{height:.1f}%",
            ha='center',
            va='bottom',
            fontsize=10
        )

    plt.tight_layout()
    plt.show()
